# 特征交叉：捕获特征间的交互效应
> 难度标签：中级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：11_特征工程 | 源文件：特征交叉.py | 核心功能：多项式特征、特征组合、交互项

## 概述

特征交叉创建原始特征的乘积或组合，捕获特征间的交互效应。例如"年龄 x 收入"可能比单独的年龄和收入更有预测力。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification

## 数学原理

### 1. 特征交叉的核心思想

线性模型无法捕捉特征间的交互关系。特征交叉通过构造特征的乘积项，使线性模型能学习非线性模式：

$$y = w_1 x_1 + w_2 x_2 + w_3 (x_1 \cdot x_2) + b$$

其中 $x_1 \cdot x_2$ 就是交叉特征，$w_3$ 捕捉两特征的交互效应。

### 2. 多项式特征展开

对原始特征 $\mathbf{x} = [x_1, x_2, \ldots, x_d]$，degree=2 的多项式展开包含：

$$\phi(\mathbf{x}) = [x_1, \ldots, x_d, x_1^2, x_1 x_2, \ldots, x_1 x_d, x_2^2, x_2 x_3, \ldots, x_d^2]$$

特征数量从 $d$ 增长到 $\binom{d+2}{2} = \frac{d(d+3)}{2}$（含偏置项）。

**仅交叉项**（`interaction_only=True`）：只保留 $x_i x_j$（$i \neq j$），不含 $x_i^2$，特征数为 $\binom{d}{2}$。

### 3. 隐式核方法视角

多项式交叉等价于核方法中的多项式核：

$$K(\mathbf{x}, \mathbf{z}) = (\mathbf{x}^\top \mathbf{z} + c)^2$$

展开后包含所有 degree $\leq 2$ 的多项式项。核技巧避免显式构造交叉特征，但线性模型显式构造更可解释。

### 4. 分类特征的交叉

对类别特征 $A \in \{a_1, a_2\}$, $B \in \{b_1, b_2, b_3\}$，交叉后生成新特征：

$$C = A \times B \in \{(a_i, b_j)\}$$

$C$ 有 $2 \times 3 = 6$ 个可能取值，捕捉 $A$ 和 $B$ 的组合效应。

代码中 `pd.get_dummies` 后做交叉，实现类别特征的笛卡尔积编码。

### 5. 维度控制

交叉特征导致维度爆炸，需要控制：
- `degree`：多项式阶数（通常 $\leq 3$）
- `interaction_only`：只保留交叉项，不含高次幂
- 特征选择后做交叉：先筛选重要特征再交叉

### 6. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `PolynomialFeatures(degree=2)` | $\phi(\mathbf{x})$：所有 degree $\leq 2$ 的项 |
| `interaction_only=True` | 只保留 $x_i x_j$，不含 $x_i^2$ |
| `include_bias=False` | 不包含常数项 1 |
| `get_feature_names_out()` | 返回特征名：$[x_1, x_2, x_1 x_2, \ldots]$ |
| `pd.get_dummies().multiply()` | 类别特征交叉编码 |
| `poly.fit_transform(X)` | $X \to \phi(X) \in \mathbb{R}^{n \times \binom{d+2}{2}}$ |

### 1. PolynomialFeatures 自动生成交叉特征

运行 1. PolynomialFeatures 自动生成交叉特征 的代码，观察算法在该环节的行为。

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

print("=" * 60)
print("1. PolynomialFeatures 自动生成交叉特征")
print("=" * 60)

# 简单数据示例
X_demo = np.array([
    [1, 2],
    [3, 4],
    [5, 6],
    [7, 8],
# --- 继续 ---
])
feature_names_base = ['面积', '房龄']

print(f"原始数据:\n{X_demo}")
print(f"特征: {feature_names_base}")

# degree=2, include_bias=False
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
X_poly = poly.fit_transform(X_demo)

print(f"\ndegree=2 多项式特征:")
feature_names_poly = poly.get_feature_names_out(feature_names_base)
for i, name in enumerate(feature_names_poly):
    print(f"  [{i}] {name}: {X_poly[:, i]}")

# 交叉项（不含平方项）
poly_interact = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
X_interact = poly_interact.fit_transform(X_demo)
feature_names_interact = poly_interact.get_feature_names_out(feature_names_base)

print(f"\n仅交叉项 (interaction_only=True):")
for i, name in enumerate(feature_names_interact):
    print(f"  [{i}] {name}: {X_interact[:, i]}")

### 2. 手动构造交互特征

运行 2. 手动构造交互特征 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("2. 手动构造交互特征")
print("=" * 60)

np.random.seed(42)
n = 100

# 模拟房价数据
area = np.random.uniform(50, 200, n)       # 面积
rooms = np.random.randint(1, 6, n)          # 房间数
age = np.random.uniform(0, 30, n)           # 房龄
district = np.random.choice(['A', 'B', 'C'], n)  # 区域

price = 5000 * area + 100000 * rooms - 3000 * age + np.random.normal(0, 50000, n)

df = pd.DataFrame({
    '面积': area,
    '房间数': rooms,
    '房龄': age,
    '区域': district,
# --- 继续 ---
    '房价': price
})

print("原始数据前5行:")
print(df.head().to_string(index=False))

# 交互特征1: 面积 / 房间数 (每间房面积)
df['每间房面积'] = df['面积'] / df['房间数']

# 交互特征2: 面积 * 房龄 (面积的老化效应)
df['面积房龄交互'] = df['面积'] * df['房龄']

# 交互特征3: 面积^2 (面积的非线性效应)
df['面积平方'] = df['面积'] ** 2

# 交互特征4: 房间数与房龄的比值
df['房间房龄比'] = df['房间数'] / (df['房龄'] + 1)

print("\n构造交互特征后:")
print(df[['面积', '房间数', '房龄', '每间房面积', '面积房龄交互', '面积平方', '房间房龄比']].head().to_string(index=False))

### 3. 分类特征交叉

在分类任务上训练模型并评估性能。

In [ ]:
print("\n" + "=" * 60)
print("3. 分类特征交叉")
print("=" * 60)

np.random.seed(42)
n2 = 200

gender = np.random.choice(['男', '女'], n2)
city = np.random.choice(['北京', '上海', '广州'], n2)
education = np.random.choice(['本科', '硕士', '博士'], n2)

# 方法1: 字符串拼接
df_cross = pd.DataFrame({
    '性别': gender,
    '城市': city,
    '学历': education,
})

# 分类交叉: 性别 + 城市
df_cross['性别_城市'] = df_cross['性别'] + '_' + df_cross['城市']

# 分类交叉: 城市 + 学历
df_cross['城市_学历'] = df_cross['城市'] + '_' + df_cross['学历']

# 三重交叉: 性别 + 城市 + 学历
df_cross['性别_城市_学历'] = df_cross['性别'] + '_' + df_cross['城市'] + '_' + df_cross['学历']

print("分类特征交叉:")
print(df_cross.head(10).to_string(index=False))

# 各组合的数量统计
print(f"\n性别_城市组合分布:")
print(df_cross['性别_城市'].value_counts().sort_index().to_string())

print(f"\n城市_学历组合分布:")
print(df_cross['城市_学历'].value_counts().sort_index().to_string())

# 方法2: 使用pd.get_dummies对交叉特征做独热编码
print("\n" + "=" * 60)
print("4. 交叉特征的独热编码")
print("=" * 60)

# 先独热编码再手动交叉
df_dummies = pd.get_dummies(df_cross[['性别', '城市']], prefix=['性别', '城市'])
print(f"独热编码后维度: {df_dummies.shape}")
print(df_dummies.head().to_string(index=False))

# 手动交叉独热编码列
cross_features = pd.DataFrame()
for g_col in [c for c in df_dummies.columns if c.startswith('性别_')]:
    for c_col in [c for c in df_dummies.columns if c.startswith('城市_')]:
        cross_name = f"{g_col}_X_{c_col}"
        cross_features[cross_name] = df_dummies[g_col] * df_dummies[c_col]

print(f"\n交叉后的独热编码维度: {cross_features.shape}")
print(cross_features.head().to_string(index=False))

### 综合总结

运行 综合总结 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("总结")
print("=" * 60)
print("特征交叉的核心思想:")
print("  1. PolynomialFeatures: 自动化，适合数值特征，注意维度爆炸")
# --- 输出结果 ---
print("  2. 手动交互特征: 业务驱动，可解释性强，需要领域知识")
print("  3. 分类特征交叉: 捕捉类别间的联合效应，注意组合数爆炸")
print("  4. 交叉后的独热编码: 维度高，适合稀疏模型或树模型")

## 关键代码解释

```python
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2, interaction_only=True)
X_cross = poly.fit_transform(X)
```

`interaction_only=True` 只创建交叉项（x1*x2），不创建高阶项（x1^2）。

## 注意事项

1. 特征数量爆炸式增长
2. 只在有意义的特征间创建交叉
3. 配合正则化使用（Lasso 自动筛选无用交叉项）

## 总结与延伸

以上代码展示了 **特征交叉** 的完整流程。

**解读要点**：
- 关注 **特征重要性** 排名和分布
- 对比特征选择前后的模型性能
- 观察特征交叉是否带来性能提升

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **FM/FFM**：推荐系统中的特征交叉模型
- **自动交叉**：Deep Crossing Network
- **业务知识驱动**：有意义的交叉通常需要领域知识
